In [ ]:
!pip -q install -U "sentence-transformers>=3.0" "transformers>=4.51.0" accelerate

import os, time, numpy as np, torch
from sentence_transformers import CrossEncoder, SentenceTransformer, util
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print(f"GPU: {torch.cuda.get_device_name(0)} | dtype: {dtype}")
else:
    device, dtype = "cpu", torch.float32
    print("WARNING: no GPU detected. This 4B model will be very slow on CPU.")

RERANKER_ID = "zeroentropy/zerank-2-reranker"
print(f"\nLoading {RERANKER_ID} (~8GB on first run)...")
reranker = CrossEncoder(
    RERANKER_ID,
    model_kwargs={"torch_dtype": dtype},
    device=device,
)
print("Reranker loaded.")

def to_prob(logits):
    return (torch.as_tensor(logits, dtype=torch.float32) / 5).sigmoid()

In [ ]:
print("\n" + "="*70 + "\nPART 1: Pairwise scoring\n" + "="*70)
pairs = [
    ("What is 2+2?", "4"),
    ("What is 2+2?", "The answer is definitely 1 million"),
    ("Which planet is the Red Planet?",
     "Mars, known for its reddish appearance, is the Red Planet."),
    ("Which planet is the Red Planet?",
     "Venus is Earth's twin because of its similar size."),
]
logits = reranker.predict(pairs, convert_to_tensor=True)
probs = to_prob(logits)
for (q, d), lg, p in zip(pairs, logits.tolist(), probs.tolist()):
    print(f"logit={lg:+6.2f}  prob={p:5.3f}  | {q[:30]:30s} -> {d[:45]}")

In [ ]:
print("\n" + "="*70 + "\nPART 2: model.rank for a single query\n" + "="*70)
query = "How do I fix a Python list index out of range error?"
candidates = [
    "IndexError happens when you access an index beyond the list length; check len() and loop bounds.",
    "Use a try/except IndexError block, or validate the index with `if i < len(lst)` before access.",
    "To install Python packages, run `pip install <package>` in your terminal.",
    "List comprehensions create new lists: `[x*2 for x in nums]`.",
    "Off-by-one errors in range(len(lst)+1) are a common cause of index out of range.",
]
ranking = reranker.rank(query, candidates, convert_to_tensor=True)
for rank, r in enumerate(ranking, 1):
    cid = r["corpus_id"]
    print(f"#{rank}  score={float(r['score']):+6.2f}  prob={to_prob(r['score']):.3f}  "
          f"| {candidates[cid][:60]}")

In [ ]:
print("\n" + "="*70 + "\nPART 3: Two-stage retrieve -> rerank pipeline\n" + "="*70)

corpus = [
    "The mitochondria is the powerhouse of the cell, producing ATP via respiration.",
    "Photosynthesis converts light energy into chemical energy in chloroplasts.",
    "ATP synthase uses a proton gradient across the inner mitochondrial membrane to make ATP.",
    "DNA replication is semi-conservative and occurs during the S phase of the cell cycle.",
    "The Krebs cycle (citric acid cycle) takes place in the mitochondrial matrix.",
    "Ribosomes translate mRNA into proteins in the cytoplasm.",
    "Glycolysis breaks glucose into pyruvate in the cytosol, yielding a net 2 ATP.",
    "The Golgi apparatus modifies, sorts, and packages proteins for secretion.",
    "Cellular respiration in mitochondria yields far more ATP than glycolysis alone.",
    "Plant cell walls are made primarily of cellulose for structural support.",
]

bi = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
corpus_emb = bi.encode(corpus, convert_to_tensor=True, normalize_embeddings=True)

def two_stage_search(q, top_k_retrieve=6, top_n_final=3):
    q_emb = bi.encode(q, convert_to_tensor=True, normalize_embeddings=True)
    hits = util.semantic_search(q_emb, corpus_emb, top_k=top_k_retrieve)[0]
    cand_ids = [h["corpus_id"] for h in hits]
    cand_docs = [corpus[i] for i in cand_ids]
    rr = reranker.rank(q, cand_docs, convert_to_tensor=True)
    out = []
    for r in rr[:top_n_final]:
        global_id = cand_ids[r["corpus_id"]]
        out.append((global_id, corpus[global_id], float(to_prob(r["score"]))))
    return cand_ids, out

q = "Where in the cell is most ATP actually produced?"
retrieved, final = two_stage_search(q)
print(f"Query: {q}\n")
print("Stage 1 (bi-encoder) top order:", retrieved)
print("\nStage 2 (zerank-2 reranked) top results:")
for gid, doc, p in final:
    print(f"  [doc {gid}] prob={p:.3f} | {doc}")

In [ ]:
print("\n" + "="*70 + "\nPART 4: NDCG@10 evaluation\n" + "="*70)

eval_set = [
    {"query": "Where is most ATP produced in the cell?",
     "rels": {0: 2, 2: 3, 4: 2, 6: 1, 8: 3}},
    {"query": "How do plants capture light energy?",
     "rels": {1: 3, 9: 1}},
    {"query": "How are proteins made and packaged in a cell?",
     "rels": {5: 3, 7: 2}},
]

def dcg(rels):
    rels = np.asarray(rels, dtype=float)
    return np.sum((2**rels - 1) / np.log2(np.arange(2, rels.size + 2)))

def ndcg_at_k(ranked_doc_ids, rel_map, k=10):
    gains = [rel_map.get(d, 0) for d in ranked_doc_ids[:k]]
    ideal = sorted(rel_map.values(), reverse=True)[:k]
    idcg = dcg(ideal)
    return dcg(gains) / idcg if idcg > 0 else 0.0

base_scores, rr_scores = [], []
for ex in eval_set:
    q, rel_map = ex["query"], ex["rels"]
    q_emb = bi.encode(q, convert_to_tensor=True, normalize_embeddings=True)
    hits = util.semantic_search(q_emb, corpus_emb, top_k=len(corpus))[0]
    base_order = [h["corpus_id"] for h in hits]
    base_scores.append(ndcg_at_k(base_order, rel_map))
    rr = reranker.rank(q, [corpus[i] for i in base_order], convert_to_tensor=True)
    rr_order = [base_order[r["corpus_id"]] for r in rr]
    rr_scores.append(ndcg_at_k(rr_order, rel_map))

print(f"{'Query':45s} {'bi-encoder':>12s} {'+ zerank-2':>12s}")
for ex, b, r in zip(eval_set, base_scores, rr_scores):
    print(f"{ex['query'][:43]:45s} {b:12.4f} {r:12.4f}")
print("-"*72)
print(f"{'AVERAGE NDCG@10':45s} {np.mean(base_scores):12.4f} {np.mean(rr_scores):12.4f}")
print(f"\nReranking lift: {np.mean(rr_scores)-np.mean(base_scores):+.4f} NDCG@10")

In [1]:
print("\n" + "="*70 + "\nPART 5: Cross-domain reranking\n" + "="*70)
domain_cases = {
    "finance": ("What does a rising debt-to-equity ratio indicate?",
        ["A higher debt-to-equity ratio means a firm is financing growth with more debt, raising financial risk.",
         "EBITDA measures operating performance before interest, taxes, depreciation and amortization.",
         "The P/E ratio compares share price to earnings per share."]),
    "legal": ("What is the difference between a misdemeanor and a felony?",
        ["Felonies are serious crimes punishable by over a year in prison; misdemeanors carry lighter penalties.",
         "A tort is a civil wrong causing harm, separate from criminal law.",
         "Habeas corpus protects against unlawful detention."]),
    "code": ("How do I reverse a string in Python?",
        ["Use slicing with a step of -1: `reversed_str = s[::-1]`.",
         "`str.join()` concatenates an iterable of strings with a separator.",
         "`list.sort()` sorts a list in place and returns None."]),
}
for domain, (q, docs) in domain_cases.items():
    best = reranker.rank(q, docs, convert_to_tensor=True)[0]
    print(f"[{domain:8s}] {q}\n  -> prob={to_prob(best['score']):.3f} | "
          f"{docs[best['corpus_id']][:70]}\n")

print("="*70 + "\nPART 6: Batched throughput\n" + "="*70)
big_query = "What organelle generates cellular energy?"
big_docs = corpus * 5
t0 = time.time()
_ = reranker.predict([(big_query, d) for d in big_docs],
                     batch_size=16, convert_to_tensor=True)
dt = time.time() - t0
print(f"Scored {len(big_docs)} pairs in {dt:.2f}s ({len(big_docs)/dt:.1f} pairs/s)")
print("\nDone. zerank-2 is non-commercial (CC-BY-NC-4.0); see the model card for licensing.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 120.8 MB/s eta 0:00:00
GPU: Tesla T4 | dtype: torch.bfloat16

Loading zeroentropy/zerank-2-reranker (~8GB on first run)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/199 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Query: Where in the cell is most ATP actually produced?

Stage 1 (bi-encoder) top order: [0, 2, 8, 6, 4, 1]

Stage 2 (zerank-2 reranked) top results:
  [doc 0] prob=0.902 | The mitochondria is the powerhouse of the cell, producing ATP via respiration.
  [doc 8] prob=0.861 | Cellular respiration in mitochondria yields far more ATP than glycolysis alone.
  [doc 2] prob=0.820 | ATP synthase uses a proton gradient across the inner mitochondrial membrane to make ATP.

PART 4: NDCG@10 evaluation
Query                                           bi-encoder   + zerank-2
Where is most ATP produced in the cell?             0.8570       0.8630
How do plants capture light energy?                 0.9828       0.9828
How are proteins made and packaged in a cel         0.8340       0.8340
------------------------------------------------------------------------
AVERAGE NDCG@10                                     0.8913       0.8933

Reranking lift: +0.0020 NDCG@10

PART 5: Cross-domain reranking
[financ